# ConvNeXt-Tiny: percentage central sampling and plane/FS attention

Separate experiment inspired by CoPAS's plane/sequence-specific fusion principles;
this is not a reproduction of its co-plane cross-sequence attention architecture.

**Sampling:** retain all usable series. Use ceil(15% × retained real slice count)
triplet centres per series, min 4/max 12, capped at the number of unique valid triplets.
Allocate approximately 70% to the middle 30% of the stack, with at least two outer
samples when capacity permits. Integer rounding and short stacks adjust this allocation.
Train centres vary without replacement; validation/inference centres are fixed interior
quantiles. Triplets overlap but are never exact duplicates within a series; only depths
below 3 repeat slices within a triplet. Real depth excludes padding. Percentages apply to
the cached depth (at most 64), not the original DICOM count; discarded slices are unavailable.
Examples: depth 30 selects 5 triplets; depth 64 selects 10. More slices increase runtime.

**Fusion:** shared ConvNeXt-Tiny → mean/max over each series → mean within each plane/FS group
→ condition-specific attention across six groups → 12 logits. Group order: sagittal FS,
sagittal non-FS, coronal FS, coronal non-FS, axial FS, axial non-FS. Missing groups receive
exactly zero attention. Duplicate acquisitions within a group do not multiply that group's
vote. Plane/FS identity is preserved through separate slots and learned group embeddings.
FS does not fully describe an MRI sequence. Fluid sensitivity remains distinct metadata;
it does not create additional groups in this experiment. All attention and classifier
parameters are included in the head optimizer group. Attention is not a reliability score.

**Labels and validation:** explicitly join transferred_labels.csv. Verified training labels
override teacher probabilities. Same seed-42 deterministic split: target 40/58 gold studies
for training and 18 official + all 20 radiologist studies for validation. Entire held-out
patient groups are excluded, including pseudo targets. Corrected loss divides weighted BCE
by active-label count (pseudo weight .25 / verified 1). No additional smoothing. Select by
combined verified BCE; report both cohorts and per-target counts separately. The same
split helper and source label hashes prevent accidental holdout changes. Teacher outputs
are not OOF and teacher training exposure to validation studies remains unverified.

**Initialization:** fresh training from the official ImageNet ConvNeXt-Tiny IMAGENET1K_V1
backbone. Attention and prediction layers are newly initialized; attention starts uniformly
over present groups. No knee-trained student weights are loaded. Optimizer, cosine schedule,
epoch counter and early-stopping state start fresh. Exact resume uses only this experiment's
own last.pt.

**Inputs:** competition, rsna-knee-normalized-all-training-data with transferred_labels.csv,
rsna-knee-radiologist-verified-20. Enable GPU and Internet for ImageNet download and model
publication, or attach the official torchvision weights via PRETRAINED_PATH. No published
student or teacher weights are required.
Start RESUME_CHECKPOINT=None. Never attach a previous model's last.pt as an exact resume.
Save/download this new last.pt before closing an interactive session. Existing inference
notebooks reject this new architecture; submission requires its group-aware input contract.
Original notebooks remain unchanged. Sampling and fusion change together in this experiment.


In [1]:
from pathlib import Path
import torch

DATASET_ROOT = None
DATASET_HANDLE = 'gany24558/rsna-knee-normalized-all-training-data'
TRANSFER_LABELS_PATH = None
RADIOLOGIST_LABELS_PATH = Path('/kaggle/input/datasets/gany24558/rsna-knee-radiologist-verified-20/radiologist_labels.csv')
COMPETITION_ROOT = None
RESUME_CHECKPOINT = None  # Only this new experiment's own complete last.pt
OUTPUT_ROOT = Path('/kaggle/working/rsna-convnext-tiny-central-group-attention')
MODEL_HANDLE = 'gany24558/gc-rsna-knee-convnext/pyTorch/central-group-attention'
REQUIRE_GPU = True
UPLOAD_MODEL = True
EXPECTED_RADIOLOGIST_STUDIES = 20
EXPECTED_OFFICIAL_GOLD_STUDIES = 58
EXPECTED_SOURCE_INDEX_SHA256 = 'e9c83cbc54c339c465823c8404eb1b96fcfebc0fe221bebf37798ffd78eb43dd'
PRETRAINED_PATH = None  # Optional official torchvision ConvNeXt-Tiny ImageNet state dictionary
CFG = dict(seed=42, gold_val_fraction=.30, pseudo_weight=.25,
           image_size=224, microbatch=8, accumulate=4, dropout=.2,
           backbone_lr=1e-5, head_lr=1e-4, epochs=35, patience=7, max_hours=7.5,
           sample_fraction=.15, min_triplets=4, max_triplets=12,
           central_sample_fraction=.70, central_depth_fraction=.30, attention_dim=128,
           sampling_policy='percentage-central-unique-triplets-v1',train_series_policy='all usable',
           loss_normalization='active-label-count-v1',group_fusion='plane-fs-condition-attention-v1')
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('Enable GPU before loading image manifests')
if RESUME_CHECKPOINT is not None:
    RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT)
    prior = torch.load(RESUME_CHECKPOINT, map_location='cpu', weights_only=False)
    init = prior['identity']['initialization']
    if init.get('mode') != 'imagenet-convnext-tiny-plane-fs-attention-v1':
        raise ValueError('Not a checkpoint from this attention experiment')
    print('Exact resume from completed epoch',prior['epoch'],flush=True)
    del prior
print('Device:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',flush=True)
print('Sampling:',CFG['sample_fraction'],'central share:',CFG['central_sample_fraction'],flush=True)


Device: Tesla T4


Sampling: 0.15 central share: 0.7


## Embedded runtime


In [2]:
import sys, json, hashlib, random
from pathlib import Path
SOURCE_FILES = {'knee_data.py': '"""Offline RSNA frozen-DINO training runtime. Embedded verbatim in the notebook."""\nimport os\nos.environ[\'HF_HUB_OFFLINE\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nimport copy, gc, hashlib, io, json, math, random, shutil, tarfile, time\nfrom contextlib import nullcontext\nfrom pathlib import Path, PurePosixPath\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import roc_auc_score, average_precision_score\n\nID, SID = \'StudyInstanceUID\', \'SeriesInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nRUNTIME_VERSION = \'rsna-frozen-dino-v1\'\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(8 << 20), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef atomic_json(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(data, indent=2, sort_keys=True, allow_nan=False, default=str))\n    tmp.replace(path)\n\n\ndef atomic_torch(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\'.tmp\'); torch.save(data, tmp); tmp.replace(path)\n\n\ndef select_one(candidates, description):\n    candidates = sorted(set(Path(p).resolve() for p in candidates))\n    if len(candidates) != 1:\n        raise ValueError(f\'Set {description} explicitly: found {len(candidates)} candidates: {candidates}\')\n    return candidates[0]\n\n\ndef find_input_files(root, filename):\n    """Discover lightweight manifests without traversing hundreds of thousands of DICOMs."""\n    for current, dirs, files in os.walk(root):\n        dirs[:] = [d for d in dirs if d not in {\'train_series\',\'test_series\',\'cache\',\'features\',\'.git\',\'__pycache__\'}]\n        if filename in files:\n            yield Path(current)/filename\n\n\ndef discover_bundle(explicit=None, input_root=\'/kaggle/input\', dataset_handle=\'gany24558/rsna-knee-normalized-all-training-data\'):\n    if explicit:\n        root = Path(explicit)\n    else:\n        candidates = []\n        for p in find_input_files(input_root,\'dataset_index.json\'):\n            try:\n                index = json.loads(p.read_text())\n                if index.get(\'dataset_handle\') == dataset_handle:\n                    candidates.append(p.parent)\n            except (ValueError, OSError): pass\n        root = select_one(candidates, \'DATASET_ROOT\')\n    if not (root / \'dataset_index.json\').is_file():\n        raise FileNotFoundError(f\'Attach {dataset_handle}, with dataset_index.json\')\n    index = json.loads((root / \'dataset_index.json\').read_text())\n    if index.get(\'dataset_handle\') != dataset_handle:\n        raise ValueError(f\'Expected dataset {dataset_handle}; found {index.get("dataset_handle")}\')\n    return root\n\n\ndef safe_member(archive, name):\n    part = PurePosixPath(name)\n    if part.is_absolute() or \'..\' in part.parts:\n        raise ValueError(f\'Unsafe archive member: {name}\')\n    member = archive.getmember(name)\n    if not member.isfile(): raise ValueError(f\'Not a regular member: {name}\')\n    return member\n\n\ndef load_bundle(root):\n    """Use only active, processed shards; never legacy archives or partial snapshots."""\n    root = Path(root); index = json.loads((root / \'dataset_index.json\').read_text())\n    identity = index[\'identity\']; expected = {f\'{i:03d}\' for i in range(int(identity[\'num_shards\']))}\n    if set(index[\'shards\']) != expected:\n        raise ValueError(f\'Preprocessing is incomplete. Missing shards: {sorted(expected-set(index["shards"]))}\')\n    frames, raw_reference, labels, prep_reference = [], None, None, None\n    for slot, entry in sorted(index[\'shards\'].items()):\n        if entry[\'status\'] != \'processed\': raise ValueError(f\'Shard {slot} is partial; resume preprocessing first\')\n        if Path(entry[\'file\']).name != entry[\'file\']: raise ValueError(\'Invalid package filename\')\n        package = root / entry[\'file\']\n        with tarfile.open(package, \'r:\') as arc:\n            info = json.load(arc.extractfile(safe_member(arc, \'publication_manifest.json\')))\n            if info[\'status\'] != \'processed\' or {k: info[k] for k in identity} != identity:\n                raise ValueError(\'Publication identity/status mismatch\')\n            prep = json.load(arc.extractfile(safe_member(arc, \'preprocessing_config.json\')))\n            if prep.get(\'run_id\') != identity[\'run_id\']: raise ValueError(\'Preprocessing config run mismatch\')\n            if prep_reference is not None and prep != prep_reference: raise ValueError(\'Preprocessing configs differ\')\n            prep_reference = prep\n            raw = arc.extractfile(safe_member(arc, \'study_labels_and_folds.csv\')).read()\n            if hashlib.sha256(raw).hexdigest() != identity[\'label_sha256\']:\n                raise ValueError(\'Label checksum mismatch\')\n            if raw_reference is not None and raw != raw_reference: raise ValueError(\'Labels differ across shards\')\n            raw_reference = raw\n            if labels is None:\n                labels = pd.read_csv(io.BytesIO(raw), dtype={ID: str, \'patient_group\': str})\n            frame = pd.read_csv(arc.extractfile(safe_member(arc, f\'training_series_shard_{int(slot):03d}.csv\')),\n                                dtype={ID: str, SID: str})\n            if len(frame) and not frame.status.eq(\'ok\').all(): raise ValueError(\'Training series contain failures\')\n            # Direct seek offsets avoid repeatedly scanning tar archives during extraction.\n            offsets, sizes = [], []\n            for member in frame.cache_path:\n                item = safe_member(arc, member); offsets.append(item.offset_data); sizes.append(item.size)\n            frame[\'archive_path\'] = str(package); frame[\'offset\'] = offsets; frame[\'nbytes\'] = sizes\n            frames.append(frame)\n    series = pd.concat(frames, ignore_index=True).sort_values([ID, SID]).reset_index(drop=True)\n    if series.empty or series.duplicated([ID, SID]).any(): raise ValueError(\'Empty or duplicated training series\')\n    if not labels[ID].is_unique: raise ValueError(\'Duplicate study labels\')\n    if not set(series[ID]).issubset(set(labels[ID])): raise ValueError(\'Series with no label record\')\n    validate_labels(labels)\n    eligible = labels.loc[labels[ID].isin(series[ID])].copy().sort_values(ID).reset_index(drop=True)\n    index = dict(index, preprocessing=prep_reference)\n    return series, eligible, labels, index\n\n\ndef validate_labels(labels):\n    required = {ID, \'fold\', \'patient_group\', \'has_gold\'}\n    required |= {t+s for t in TARGETS for s in [\'\', \'__mask\', \'__weight\', \'__gold\']}\n    if required - set(labels): raise ValueError(f\'Missing label columns: {sorted(required-set(labels))}\')\n    if labels.patient_group.isna().any() or labels.patient_group.str.strip().eq(\'\').any():\n        raise ValueError(\'Missing grouping identity\')\n    folds = pd.to_numeric(labels.fold, errors=\'raise\').to_numpy(float)\n    if not np.isfinite(folds).all() or not np.equal(folds, folds.astype(int)).all() or (folds < -1).any():\n        raise ValueError(\'Invalid fold values\')\n    if labels.groupby(\'patient_group\').fold.nunique().gt(1).any(): raise ValueError(\'A group crosses folds\')\n    for t in TARGETS:\n        for s in [\'__mask\', \'__gold\']:\n            if not labels[t+s].isin([0, 1]).all(): raise ValueError(\'Invalid label mask\')\n        m = labels[t+\'__mask\'].eq(1); g = labels[t+\'__gold\'].eq(1)\n        w = labels[t+\'__weight\'].to_numpy(float)\n        if not np.isfinite(w).all() or (w < 0).any(): raise ValueError(\'Invalid reliability weights\')\n        values = labels.loc[m, t].to_numpy(float)\n        if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():\n            raise ValueError(\'Known target must be a finite probability in [0, 1]\')\n        if not labels.loc[g, t].isin([0, 1]).all(): raise ValueError(\'Official target must be binary\')\n        if (g & ~m).any() or (g & labels[t+\'__weight\'].ne(1)).any(): raise ValueError(\'Invalid verified override\')\n    any_gold = labels[[t+\'__gold\' for t in TARGETS]].any(axis=1)\n    if not labels.has_gold.eq(any_gold.astype(int)).all(): raise ValueError(\'has_gold does not match per-target masks\')\n    if labels.loc[any_gold, \'fold\'].lt(0).any(): raise ValueError(\'Verified studies need validation folds\')\n\n\ndef smooth_soft_targets(target, epsilon=0.05):\n    """Smooth probabilities once; missing values remain missing (not negatives)."""\n    if not 0 <= epsilon < 0.5:\n        raise ValueError(\'epsilon must be in [0, 0.5)\')\n    return target * (1 - 2 * epsilon) + epsilon\n\n\ndef apply_community_labels(labels, path, epsilon=0.05, weight=0.25, column_map=None):\n    """Replace generated labels only. Keep official labels, folds and groups unchanged.\n\n    column_map maps our target names to parquet column names. No positional guesses.\n    """\n    path = Path(path)\n    if not path.is_file():\n        raise FileNotFoundError(f\'Attach rsna-knee-hpo-assets and set COMMUNITY_LABELS_PATH: {path}\')\n    if not np.isfinite(weight) or weight <= 0:\n        raise ValueError(\'Community label weight must be positive and finite\')\n    frame = pd.read_parquet(path)\n    if ID not in frame and frame.index.name == ID:\n        frame = frame.reset_index()\n    mapping = column_map or {t: t for t in TARGETS}\n    if set(mapping) != set(TARGETS) or len(set(mapping.values())) != len(TARGETS):\n        raise ValueError(\'Provide one distinct community column per target\')\n    missing = {ID, *mapping.values()} - set(frame.columns)\n    if missing:\n        raise ValueError(f\'Missing parquet columns {sorted(missing)}; available: {list(frame.columns)}. Set COMMUNITY_COLUMN_MAP explicitly.\')\n    if frame[ID].isna().any(): raise ValueError(\'Missing community study ID\')\n    frame[ID] = frame[ID].astype(str)\n    if frame[ID].duplicated().any(): raise ValueError(\'Duplicate community study IDs\')\n    extra = set(frame[ID]) - set(labels[ID])\n    if extra: raise ValueError(f\'Community labels include {len(extra)} IDs outside official training data\')\n    if not len(frame): raise ValueError(\'Empty community label table\')\n    frame = frame.set_index(ID)\n    out = labels.copy()\n    for t in TARGETS:\n        values = pd.to_numeric(frame[mapping[t]], errors=\'raise\').astype(float)\n        present = values.notna()\n        if not np.isfinite(values[present]).all() or not values[present].between(0, 1).all():\n            raise ValueError(f\'Invalid community probabilities: {t}\')\n        aligned = values.reindex(out[ID]).to_numpy(float)\n        gold = out[t+\'__gold\'].eq(1).to_numpy()\n        known = np.isfinite(aligned) & ~gold\n        out[t] = out[t].astype(float)\n        out.loc[~gold, t] = smooth_soft_targets(aligned[~gold], epsilon)\n        out.loc[~gold, t+\'__mask\'] = known[~gold].astype(int)\n        out.loc[~gold, t+\'__weight\'] = np.where(known[~gold], weight, 0.)\n        out.loc[~gold, t+\'__source\'] = np.where(known[~gold], \'community_soft\', \'none\')\n        out.loc[~gold, t+\'__state\'] = np.where(known[~gold], \'soft\', \'unmentioned\')\n    validate_labels(out)\n    if not out[[t+\'__mask\' for t in TARGETS]].to_numpy(bool)[~out.has_gold.astype(bool)].any():\n        raise ValueError(\'No usable community targets for non-gold studies\')\n    provenance = dict(filename=path.name, sha256=file_hash(path), epsilon=epsilon,\n                      weight=weight, column_map=mapping, rows=len(frame),\n                      missing_studies=len(set(labels[ID])-set(frame.index)))\n    return out, provenance\n\n\ndef audit_labels(labels):\n    rows = []\n    for fold, block in labels.groupby(\'fold\'):\n        for t in TARGETS:\n            m = block[t+\'__mask\'].eq(1) & block[t+\'__weight\'].gt(0)\n            g = block[t+\'__gold\'].eq(1)\n            rows.append(dict(fold=int(fold), target=t, studies=len(block), positive=int((m & block[t].eq(1)).sum()),\n                             negative=int((m & block[t].eq(0)).sum()), soft=int((m & block[t].gt(0) & block[t].lt(1)).sum()), unknown=int((~m).sum()),\n                             verified_positive=int((g & block[t].eq(1)).sum()),\n                             verified_negative=int((g & block[t].eq(0)).sum())))\n    return pd.DataFrame(rows)\n\n\ndef read_native(record, expected_run):\n    with open(record[\'archive_path\'], \'rb\') as f:\n        f.seek(int(record[\'offset\'])); payload = f.read(int(record[\'nbytes\']))\n    if len(payload) != int(record[\'nbytes\']): raise ValueError(\'Truncated archive\')\n    with np.load(io.BytesIO(payload), allow_pickle=False) as z:\n        image = z[\'image\']; meta = json.loads(str(z[\'meta\'].item()))\n        support = np.unpackbits(z[\'valid_bits\'], count=image.size).reshape(image.shape).astype(bool)\n    if image.dtype != np.float16 or image.shape != (64, 320, 320) or list(image.shape) != meta[\'shape\']:\n        raise ValueError(\'Expected preprocessing contract float16 [64,320,320]\')\n    if meta.get(\'representation\') != \'native-plane-full-fov-v3\' or meta.get(\'run_id\') != expected_run:\n        raise ValueError(\'Wrong normalization identity/representation\')\n    if meta.get(ID) != record[ID] or meta.get(SID) != record[SID]: raise ValueError(\'Cache ID mismatch\')\n    if not np.isfinite(image).all() or image.min() < 0 or image.max() > 1: raise ValueError(\'Invalid pixel range\')\n    n = int(meta[\'selected_count\'])\n    if n <= 0 or n > len(image) or support[n:].any(): raise ValueError(\'Invalid acquired-slice mask\')\n    valid = np.flatnonzero(support[:n].reshape(n, -1).any(axis=1))\n    positions = np.asarray(meta[\'slice_positions_mm\'], dtype=np.float32)\n    spacing = np.asarray(meta[\'spacing_summary_drc_mm\'], dtype=np.float32)\n    if len(valid) != n or len(positions) != n or not np.isfinite(positions).all(): raise ValueError(\'Invalid slice positions\')\n    if n > 1 and not (np.diff(positions) > 0).all(): raise ValueError(\'Unordered slice positions\')\n    if spacing.shape != (3,) or not np.isfinite(spacing).all() or (spacing <= 0).any(): raise ValueError(\'Invalid spacing\')\n    return image, support, meta\n\n\ndef evaluate_metrics(y,p,gold):\n    rows=[]\n    for t,name in enumerate(TARGETS):\n        v=gold[:,t];truth=y[v,t];score=p[v,t];positive=int((truth==1).sum());negative=int((truth==0).sum())\n        auc=float(roc_auc_score(truth,score)) if positive and negative else None\n        ap=float(average_precision_score(truth,score)) if positive and negative else None\n        rows.append(dict(target=name,n=int(v.sum()),positive=positive,negative=negative,auroc=auc,average_precision=ap))\n    aucs=[r[\'auroc\'] for r in rows if r[\'auroc\'] is not None]\n    return dict(macro_auroc=float(np.mean(aucs)) if aucs else None,targets=rows,\n                note=\'Average precision uses sklearn; it is not trapezoidal PR-AUC.\')\n\n', 'cnn_runtime.py': '"""End-to-end ConvNeXt-Tiny study-level MRI training."""\nfrom pathlib import Path\nimport copy, json, time, hashlib, random\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\nimport knee_data as data\nfrom plane_fs_attention import ConvNeXtKnee, sample_triplet_indices, group_index, GROUPS\n\n\nclass EarlyStop:\n    def __init__(self,patience=3):\n        self.patience=patience;self.best=float(\'inf\');self.bad_epochs=0\n    def update(self,loss):\n        if not np.isfinite(loss):raise ValueError(\'Nonfinite validation loss\')\n        improved=loss<self.best\n        if improved:self.best=float(loss);self.bad_epochs=0\n        else:self.bad_epochs+=1\n        return improved,self.bad_epochs>=self.patience\n\ndef split_studies(labels,val_fraction=.2,seed=42):\n    from sklearn.model_selection import GroupShuffleSplit\n    data.validate_labels(labels)\n    splitter=GroupShuffleSplit(n_splits=1,test_size=val_fraction,random_state=seed)\n    train_ix,val_ix=next(splitter.split(labels,groups=labels.patient_group))\n    held=set(labels.iloc[val_ix].patient_group)\n    known=labels[[t+\'__mask\' for t in data.TARGETS]].to_numpy(bool)\n    weights=labels[[t+\'__weight\' for t in data.TARGETS]].to_numpy(float)\n    active=(known & (weights>0)).any(1)\n    train=labels.iloc[train_ix].loc[active[train_ix]].copy()\n    val=labels.iloc[val_ix].loc[labels.iloc[val_ix].has_gold.eq(1)].copy()\n    if train.empty or val.empty:raise ValueError(\'Need nonempty training and verified validation studies; inspect split\')\n    if set(train.patient_group)&held:raise ValueError(\'Group leakage\')\n    return train.reset_index(drop=True),val.reset_index(drop=True),held\n\n\ndef prepare_series(image,support,meta,cfg,rng=None):\n    n=int(meta[\'selected_count\'])\n    if n > len(image):raise ValueError(\'Real slice count exceeds image depth\')\n    neighbors=sample_triplet_indices(n,cfg,rng)\n    x=torch.from_numpy(image[neighbors].astype(np.float32))\n    x=F.interpolate(x,size=(cfg[\'image_size\'],cfg[\'image_size\']),mode=\'bilinear\',align_corners=False,antialias=True)\n    # Full field of view, no flips or crops that might change anatomical interpretation.\n    if rng is not None:\n        x=(x*float(rng.uniform(.9,1.1))).clamp(0,1)\n    return x\n\ndef study_images(uid,by_study,run_id,cfg,rng=None):\n    images=[]\n    for record in sorted(by_study[uid], key=lambda r:str(r[data.SID])):\n        image,support,meta=data.read_native(record,run_id)\n        images.append(dict(images=prepare_series(image,support,meta,cfg,rng),group=group_index(record)))\n    return images\n\ndef row_targets(row,verified=False):\n    y=torch.tensor([[0 if pd.isna(row[t]) else float(row[t]) for t in data.TARGETS]],dtype=torch.float32)\n    mask=torch.tensor([[bool(row[t+(\'__gold\' if verified else \'__mask\')]) for t in data.TARGETS]])\n    w=torch.ones_like(y) if verified else torch.tensor([[float(row[t+\'__weight\']) for t in data.TARGETS]])\n    return y,mask,w\n\ndef masked_loss(logits,y,mask,weights):\n    w=mask.to(logits.dtype)*weights\n    if not (w>0).any():raise ValueError(\'Study has no supervised targets\')\n    return (F.binary_cross_entropy_with_logits(logits,y,reduction=\'none\')*w).sum()/mask.to(logits.dtype).sum()\n\nclass BudgetReached(Exception):pass\n\ndef check_budget(deadline):\n    if time.monotonic()>deadline:raise BudgetReached(\'Session time budget reached\')\n\n@torch.inference_mode()\ndef validate(model,val,by_study,run_id,cfg,device,deadline):\n    model.eval();zs=[];ys=[];masks=[]\n    for row in val.to_dict(\'records\'):\n        check_budget(deadline)\n        images=study_images(row[data.ID],by_study,run_id,cfg)\n        with torch.autocast(device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n            z=model(images,cfg[\'microbatch\'])\n        zs.append(z.float().cpu());y,m,_=row_targets(row,True);ys.append(y);masks.append(m)\n    z,y,m=map(torch.cat,(zs,ys,masks))\n    if not torch.isfinite(z).all():raise ValueError(\'Nonfinite validation predictions\')\n    # Compute one loss over the full verified set, not a mean of unequal batch means.\n    loss=masked_loss(z,y,m,torch.ones_like(y)).item()\n    metrics=data.evaluate_metrics(y.numpy(),z.sigmoid().numpy(),m.numpy())\n    if \'validation_cohort\' in val:\n        cohort_metrics = {}\n        for cohort in sorted(val.validation_cohort.unique()):\n            take = torch.from_numpy(val.validation_cohort.eq(cohort).to_numpy())\n            cz, cy, cm = z[take], y[take], m[take]\n            block = data.evaluate_metrics(cy.numpy(), cz.sigmoid().numpy(), cm.numpy())\n            block[\'bce\'] = masked_loss(cz, cy, cm, torch.ones_like(cy)).item()\n            block[\'studies\'] = int(take.sum())\n            cohort_metrics[cohort] = block\n            print(\'Validation cohort:\', cohort, \'studies:\', block[\'studies\'],\n                  \'BCE:\', block[\'bce\'], \'AUROC:\', block[\'macro_auroc\'], flush=True)\n        metrics[\'cohorts\'] = cohort_metrics\n    return loss,metrics,z.sigmoid().numpy()\n\ndef train_model(model,train,val,series,run_id,cfg,work,identity,resume=None,initial_best=None):\n    work=Path(work);work.mkdir(parents=True,exist_ok=True)\n    device=next(model.parameters()).device\n    by={uid:g.sort_values(data.SID).to_dict(\'records\') for uid,g in series.groupby(data.ID)}\n    optimizer=torch.optim.Adam([{\'params\':model.backbone.parameters(),\'lr\':cfg[\'backbone_lr\']},\n                                {\'params\':model.head.parameters(),\'lr\':cfg[\'head_lr\']}])\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_0=10,T_mult=1,eta_min=1e-6)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n    stop=EarlyStop(cfg[\'patience\']);rng=np.random.default_rng(cfg[\'seed\']);history=[];start=0;best=None\n    if initial_best is not None and not resume:\n        best=copy.deepcopy(initial_best)\n        stop.best=best[\'val_loss\']\n        data.atomic_torch(work/\'best.pt\',best)\n    if resume:\n        saved=torch.load(resume,map_location=device,weights_only=False) # only your own trusted checkpoint\n        if saved[\'identity\']!=identity:raise ValueError(\'Resume identity/config/data mismatch\')\n        model.load_state_dict(saved[\'state\']);optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\']);scaler.load_state_dict(saved[\'scaler\'])\n        rng.bit_generator.state=saved[\'rng\'];torch.set_rng_state(saved[\'torch_rng\'].cpu())\n        if device.type==\'cuda\':torch.cuda.set_rng_state_all([s.cpu() for s in saved[\'cuda_rng\']])\n        history=saved[\'history\'];start=saved[\'epoch\'];best=saved[\'best\']\n        stop.best=best[\'val_loss\'];stop.bad_epochs=saved[\'bad_epochs\']\n    deadline=time.monotonic()+cfg[\'max_hours\']*3600;reason=\'max_epochs\'\n    try:\n        if stop.bad_epochs>=stop.patience:reason=\'early_stopping\'\n        else:\n            for epoch in range(start,cfg[\'epochs\']):\n                model.train();records=train.to_dict(\'records\');order=rng.permutation(len(records));losses=[]\n                optimizer.zero_grad(set_to_none=True)\n                for step,idx in enumerate(order):\n                    check_budget(deadline)\n                    row=records[idx];images=study_images(row[data.ID],by,run_id,cfg,rng)\n                    y,m,w=[v.to(device) for v in row_targets(row)]\n                    window_start=(step//cfg[\'accumulate\'])*cfg[\'accumulate\']\n                    window_size=min(cfg[\'accumulate\'],len(order)-window_start)\n                    with torch.autocast(device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n                        logits=model(images,cfg[\'microbatch\']);loss=masked_loss(logits,y,m,w)\n                    if not torch.isfinite(loss):raise ValueError(\'Nonfinite training loss\')\n                    scaler.scale(loss/window_size).backward();losses.append(float(loss.detach()))\n                    if (step+1)%cfg[\'accumulate\']==0 or step+1==len(order):\n                        scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n                        scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)\n                    if (step+1)%100==0:print(f\'Epoch {epoch+1}: {step+1}/{len(order)} studies\',flush=True)\n                val_loss,metrics,pred=validate(model,val,by,run_id,cfg,device,deadline)\n                improved,done=stop.update(val_loss)\n                row=dict(epoch=epoch+1,train_loss=float(np.mean(losses)),val_loss=val_loss,\n                         macro_auroc=metrics[\'macro_auroc\'],bad_epochs=stop.bad_epochs)\n                row[\'backbone_lr\'] = optimizer.param_groups[0][\'lr\']\n                row[\'head_lr\'] = optimizer.param_groups[1][\'lr\']\n                for cohort, values in metrics.get(\'cohorts\', {}).items():\n                    row[cohort+\'_val_loss\'] = values[\'bce\']\n                    row[cohort+\'_macro_auroc\'] = values[\'macro_auroc\']\n                history.append(row);print(row,flush=True)\n                if improved:\n                    best=dict(state_dict={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},\n                              epoch=epoch+1,val_loss=val_loss,metrics=metrics)\n                    data.atomic_torch(work/\'best.pt\',best)\n                    frame=pd.DataFrame(pred,columns=data.TARGETS);frame.insert(0,data.ID,val[data.ID].tolist())\n                    frame.to_csv(work/\'validation_predictions_private.csv\',index=False)\n                scheduler.step(epoch+1)\n                pd.DataFrame(history).to_csv(work/\'history.csv\',index=False)\n                data.atomic_torch(work/\'last.pt\',dict(identity=identity,state=model.state_dict(),optimizer=optimizer.state_dict(),\n                    scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),epoch=epoch+1,best=best,\n                    bad_epochs=stop.bad_epochs,history=history,rng=rng.bit_generator.state,\n                    torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all() if device.type==\'cuda\' else []))\n                if done:reason=\'early_stopping\';break\n    except BudgetReached:\n        reason=\'time_budget\';print(\'Time budget reached; resume from last.pt (last fully validated epoch).\',flush=True)\n    if best is None:raise RuntimeError(\'No completed validated epoch; no model will be exported or uploaded\')\n    model.load_state_dict(best[\'state_dict\']);model.eval()\n    data.atomic_torch(work/\'best.pt\', best)\n    pd.DataFrame(history).to_csv(work/\'history.csv\', index=False)\n    data.atomic_json(work/\'training_status.json\',dict(reason=reason,best_epoch=best[\'epoch\'],best_val_loss=best[\'val_loss\']))\n    return best,history,reason\n\ndef export_model(model,best,cfg,index,identity,work,source_files):\n    package=Path(work)/\'model_package\';package.mkdir(exist_ok=True)\n    data.atomic_torch(package/\'model.pt\',dict(state_dict=best[\'state_dict\'],architecture=\'convnext-tiny-plane-fs-attention-v1\',dropout=cfg[\'dropout\']))\n    for name,text in source_files.items():(package/name).write_text(text)\n    data.atomic_json(package/\'preprocessing_config.json\',index[\'preprocessing\'])\n    # Validate saved weights on synthetic image bags without training records.\n    loaded=ConvNeXtKnee(pretrained=False,dropout=cfg[\'dropout\'],attention_dim=cfg[\'attention_dim\']).eval()\n    loaded.load_state_dict(torch.load(package/\'model.pt\',weights_only=True,map_location=\'cpu\')[\'state_dict\'],strict=True)\n    device=next(model.parameters()).device\n    gen=torch.Generator().manual_seed(71)\n    images=[dict(images=torch.rand(k,3,cfg[\'image_size\'],cfg[\'image_size\'],generator=gen),group=g)\n            for k,g in [(2,0),(3,2),(1,2),(2,5)]]\n    with torch.inference_mode():\n        expected=loaded(images).cpu();actual=model(images).float().cpu()\n    torch.testing.assert_close(actual,expected,rtol=1e-3,atol=1e-4)\n    data.atomic_torch(package/\'synthetic_smoke.pt\',dict(images=images,expected=expected))\n    import importlib.metadata\n    versions={k:importlib.metadata.version(k) for k in [\'torch\',\'torchvision\',\'numpy\',\'pandas\',\'scipy\',\'pydicom\',\'scikit-learn\']}\n    (package/\'requirements.txt\').write_text(\'\\n\'.join(f\'{k}=={v}\' for k,v in versions.items())+\'\\n\')\n    (package/\'README.md\').write_text(\'ConvNeXt-Tiny CNN knee MRI model. Load model.pt with cnn_runtime.ConvNeXtKnee(pretrained=False, attention_dim=configuration[attention_dim]).\\n\'\n        \'See manifest.json for native preprocessing, triplet slices, resize, normalization and aggregation.\\n\'\n        \'Requires group-aware inference; old mean-series and DINOv2 notebooks are incompatible. Keep private. No labels or patient rows included.\\n\')\n    manifest=dict(schema_version=1,model_family=\'convnext-tiny-plane-fs-attention-v1\',targets=data.TARGETS,configuration=cfg,\n        training_identity=identity,best_epoch=best[\'epoch\'],best_validation_loss=best[\'val_loss\'],\n        validation_metrics=best[\'metrics\'],environment=versions,\n        inference=dict(representation=\'adjacent-slice-triplet\',resize=\'bilinear antialias full-FOV\',\n        image_size=cfg[\'image_size\'],sampling=\'percentage-central-unique-triplets-v1\',\n        sample_fraction=cfg[\'sample_fraction\'],min_triplets=cfg[\'min_triplets\'],max_triplets=cfg[\'max_triplets\'],\n        central_sample_fraction=cfg[\'central_sample_fraction\'],central_depth_fraction=cfg[\'central_depth_fraction\'],\n        boundary=\'unique inward triplets; repeat only when n<3\',attention_dim=cfg[\'attention_dim\'],\n        group_order=[dict(plane=p,fat_suppression=fs) for p,fs in GROUPS],\n        series_selection=\'all usable, sorted SeriesInstanceUID\',slice_pool=\'mean+max\',series_pool=\'mean within plane/FS group\',group_fusion=\'condition-specific attention; missing groups masked\',\n        outputs=\'12 logits; sigmoid once\',normalization_mean=[.485,.456,.406],normalization_std=[.229,.224,.225]),\n        files={str(p.relative_to(package)):data.file_hash(p) for p in package.rglob(\'*\') if p.is_file() and p.name!=\'manifest.json\'})\n    data.atomic_json(package/\'manifest.json\',manifest)\n    return package\n', 'native_preprocessing.py': 'from pathlib import Path\nimport hashlib, json, time, os, shutil, itertools\nimport numpy as np\nfrom scipy.ndimage import affine_transform, gaussian_filter\nimport pydicom\nfrom pydicom.pixels import apply_modality_lut\nID=\'StudyInstanceUID\'\nSID=\'SeriesInstanceUID\'\nAXES = {\n    \'Sagittal\': np.array([[1,0,0], [0,0,1], [0,-1,0]], float),\n    \'Coronal\': np.array([[0,0,1], [1,0,0], [0,-1,0]], float),\n    \'Axial\': np.array([[0,0,1], [0,1,0], [1,0,0]], float),\n}\ndef validate_and_extract_geometry(headers, expected_study, expected_series, plane, cfg):\n    """\n    Validates DICOM header geometry, checks for spatial consistency, checks \n    slice-spacing consistency within configured tolerances, and extracts the physical coordinate basis matrix.\n    \n    Parameters:\n        headers (list): List of pydicom dataset headers for a single series.\n        expected_study (str): Expected StudyInstanceUID for validation.\n        expected_series (str): Expected SeriesInstanceUID for validation.\n        plane (str): Expected anatomical plane (\'Sagittal\', \'Coronal\', \'Axial\').\n        cfg (dict): Pipeline configuration dictionary containing tolerances.\n        \n    Returns:\n        dict: Spatial metadata including slice order, origin, basis matrix, and warnings.\n    """\n    if plane not in AXES:\n        raise ValueError(f"Unknown plane: {plane}")\n    if len(headers) < 2:\n        raise ValueError("Need at least two spatial slices")\n\n    first = headers[0]\n    \n    # iop is an array of 6 floating-point numbers (standard medical imaging metadata):\n    # - The first 3 values (iop[:3]): Direction cosines for the image row.\n    # - The last 3 values (iop[3:]): Direction cosines for the image column.\n    iop = np.asarray(first.ImageOrientationPatient, float)\n    \n    spacing = np.asarray(first.PixelSpacing, float)\n\n    # 1. Validate initial orientation and spacing parameters\n    if iop.shape != (6,) or not np.isfinite(iop).all():\n        raise ValueError("Invalid orientation")\n    if spacing.shape != (2,) or not np.isfinite(spacing).all() or min(spacing) <= 0:\n        raise ValueError("Invalid pixel spacing")\n\n    col, row = iop[:3], iop[3:]\n    if (\n        abs(np.linalg.norm(col) - 1) > 1e-3\n        or abs(np.linalg.norm(row) - 1) > 1e-3\n        or abs(col @ row) > 1e-3\n    ):\n        raise ValueError("Orientation vectors are not orthonormal")\n\n    normal = np.cross(col, row)\n    photo = str(first.PhotometricInterpretation)\n    if photo not in {"MONOCHROME1", "MONOCHROME2"}:\n        raise ValueError("Only monochrome MRI supported")\n\n    shape = (int(first.Rows), int(first.Columns))\n    positions = []\n    sops = []\n\n    # 2. Iterate and validate consistency across all slices in the series\n    for h in headers:\n        if str(h.StudyInstanceUID) != expected_study or str(h.SeriesInstanceUID) != expected_series:\n            raise ValueError("DICOM study/series ID mismatch")\n        if int(getattr(h, "NumberOfFrames", 1)) != 1:\n            raise ValueError("Enhanced/multiframe DICOM unsupported")\n        if int(getattr(h, "SamplesPerPixel", 1)) != 1:\n            raise ValueError("Non-grayscale DICOM")\n        if (int(h.Rows), int(h.Columns)) != shape:\n            raise ValueError("Mixed dimensions within series")\n        if not np.allclose(h.ImageOrientationPatient, iop, atol=1e-4, rtol=0):\n            raise ValueError("Mixed orientations within series")\n        if not np.allclose(h.PixelSpacing, spacing, atol=1e-4, rtol=0):\n            raise ValueError("Mixed pixel spacing within series")\n        if str(h.PhotometricInterpretation) != photo:\n            raise ValueError("Mixed photometric interpretation")\n        \n        positions.append(np.asarray(h.ImagePositionPatient, float))\n        sops.append(str(h.SOPInstanceUID))\n\n    if len(set(sops)) != len(sops):\n        raise ValueError("Duplicate SOP instances")\n\n    positions = np.asarray(positions)\n    if positions.shape != (len(headers), 3) or not np.isfinite(positions).all():\n        raise ValueError("Missing/nonfinite slice position")\n\n    # 3. Sort slice positions and evaluate spatial gaps\n    order = np.argsort(positions @ normal, kind="stable")\n    positions = positions[order]\n    distances = positions @ normal\n    gaps = np.diff(distances)\n    dz = float(np.median(gaps))\n\n    if dz <= 0 or min(gaps) < 1e-3:\n        raise ValueError("Duplicate/non-spatial slice positions")\n    if np.max(abs(gaps - dz)) > max(0.05, cfg["gap_tolerance"] * dz):\n        raise ValueError("Irregular slice spacing / missing slices; not silently interpolated")\n\n    residual = positions - (positions[0] + np.arange(len(headers))[:, None] * dz * normal)\n    if np.linalg.norm(residual, axis=1).max() > 0.2:\n        raise ValueError("Nonparallel / shifted slice stack")\n\n    # 4. Compute anatomical obliquity and flag warnings if needed\n    angle = float(np.degrees(np.arccos(np.clip(abs(normal @ AXES[plane][:, 0]), 0, 1))))\n    nearest = min(AXES, key=lambda p: np.degrees(np.arccos(np.clip(abs(normal @ AXES[p][:, 0]), 0, 1))))\n    \n    review_warning = ""\n    if angle > cfg["max_obliquity_degrees"]:\n        review_warning = (\n            f"Orientation review: {angle:.1f} degrees from listed {plane}; "\n            f"warning threshold {cfg[\'max_obliquity_degrees\']:.1f}; nearest plane {nearest}. "\n            "Native acquisition retained; plane label unchanged."\n        )\n\n    # 5. Construct the final 3D coordinate basis matrix\n    basis = np.column_stack([normal * dz, row * spacing[0], col * spacing[1]])\n    \n    return dict(\n        order=order, origin=positions[0], basis=basis,\n        shape=(len(headers), *shape), photo=photo, angle=angle, dz=dz,\n        review_warning=review_warning, nearest_plane=nearest,\n        slice_thickness=float(getattr(first, "SliceThickness", 0) or 0)\n    )\n\n\ndef native_grid(g, plane, cfg):\n    """\n    Performs spatial normalization on individual scan slices. It aligns, scales, \n    and computes the 2D affine transformation matrices needed to map heterogeneous \n    scanner layouts onto a uniform, standardized output grid without arbitrary 3D rotations.\n    \n    Parameters:\n        g (dict): Spatial metadata dictionary extracted from geometry validation.\n        plane (str): Expected anatomical plane (\'Sagittal\', \'Coronal\', \'Axial\').\n        cfg (dict): Pipeline configuration dictionary containing target size and spacing.\n        \n    Returns:\n        tuple: (matrix, offset, basis, crop) used for downstream image resampling.\n    """\n    # 1. Select native in-plane axes without through-plane interpolation or arbitrary rotations\n    native = g[\'basis\'][:, 1:]\n    unit = native / np.linalg.norm(native, axis=0)\n    \n    # 2. Test valid axis permutations and sign flips to match the target anatomical orientation\n    choices = []\n    for perm in [(0, 1), (1, 0)]:\n        for signs in itertools.product([-1, 1], repeat=2):\n            directions = unit[:, perm] * np.array(signs)\n            score = float(np.sum(directions * AXES[plane][:, 1:]))\n            choices.append((score, directions))\n            \n    directions = max(choices, key=lambda x: x[0])[1]\n    shape = np.array(g[\'shape\'][1:])\n    \n    # 3. Calculate physical dimensions (extent) of the scan field of view (FOV)\n    corners = np.array(list(itertools.product(*[(0, n - 1) for n in shape])))\n    projections = (corners @ native.T) @ directions\n    extent = np.ptp(projections, axis=0)\n    \n    # 4. Determine pixel spacing, optionally scaling to fit the full FOV while preserving aspect ratio\n    spacing = float(cfg[\'spacing_mm\'])\n    if cfg.get(\'fit_full_fov\', False):\n        spacing = max(spacing, float(extent.max()) / (cfg[\'size\'] - 1))\n        \n    # 5. Compute the 2D affine transformation matrix:\n    #    The resampling matrix maps output pixel coordinates back to native source coordinates \n    #    using scaling, axis swaps and flips; offset supplies the translation. No arbitrary rotation is used.\n    basis = directions * spacing\n    matrix = np.linalg.pinv(native) @ basis\n    matrix[np.abs(matrix) < 1e-10] = 0\n    matrix = np.where(np.abs(matrix - np.rint(matrix)) < 1e-10, np.rint(matrix), matrix)\n    \n    # 6. Calculate centering offsets (translation part of the spatial alignment)\n    center = (shape - 1) / 2\n    offset = center - matrix @ np.full(2, (cfg[\'size\'] - 1) / 2)\n    offset[np.abs(offset) < 1e-10] = 0\n    offset = np.where(np.abs(offset - np.rint(offset)) < 1e-10, np.rint(offset), offset)\n    \n    # 7. Validate safety constraints to ensure excessive cropping does not occur\n    crop = np.maximum(0, 1 - (cfg[\'size\'] - 1) * spacing / np.maximum(extent, 1e-9))\n    if crop.max() > cfg[\'max_crop_fraction\'] + 1e-8:\n        raise ValueError(f\'Native FOV crop would be {crop.max():.1%}; increase size/spacing\')\n        \n    return matrix, offset, basis, crop\n\n\ndef decode_native_slice(path):\n    """\n    Loads an individual DICOM slice file, masks out background padding values, \n    applies the DICOM modality LUT or rescale parameters, and returns the processed float32 \n    image alongside a validity mask.\n    \n    Parameters:\n        path (str or Path): File path to the DICOM slice.\n        \n    Returns:\n        tuple: (image, valid_mask)\n            - image (np.ndarray): 32-bit floating-point array after modality scaling; MRI values need not have standardized physical units.\n            - valid_mask (np.ndarray): Boolean mask filtering out padding and non-finite values.\n    """\n    # 1. Load the DICOM file and validate that the pixel array is strictly 2D\n    ds = pydicom.dcmread(path)\n    raw = ds.pixel_array\n    if raw.ndim != 2:\n        raise ValueError("Expected single-frame 2D pixels")\n        \n    # 2. Mask explicit DICOM padding values; this does not identify all air or anatomical background\n    valid = np.ones(raw.shape, bool)\n    if hasattr(ds, "PixelPaddingValue"):\n        a = float(ds.PixelPaddingValue)\n        b = float(getattr(ds, "PixelPaddingRangeLimit", a))\n        valid &= (raw < min(a, b)) | (raw > max(a, b))\n        \n    # 3. Apply the modality LUT or rescale slope/intercept when supplied by the DICOM metadata\n    image = apply_modality_lut(raw, ds).astype(np.float32)\n    \n    # 4. Return the calibrated image and combine the padding mask with a finite-value check\n    return image, valid & np.isfinite(image)\n\n\ndef series_intensity_limits(paths, cfg):\n    """\n    Calculates robust global pixel intensity clipping bounds across an entire series \n    of DICOM slices using a memory-efficient, deterministic sampling strategy.\n    \n    Parameters:\n        paths (list): List of file paths to the DICOM slices in the series.\n        cfg (dict): Pipeline configuration containing sampling targets and percentile definitions.\n        \n    Returns:\n        tuple: (low, high) floating-point intensity limits for outlier clipping.\n    """\n    # 1. Determine a memory-efficient pixel sample budget per slice to prevent RAM spikes\n    per_slice = max(1, cfg[\'intensity_sample_pixels\'] // len(paths))\n    samples = []\n    \n    # 2. Iterate through each slice, decode, and extract evenly-spaced valid pixel samples\n    for path in paths:\n        image, valid = decode_native_slice(path)\n        values = image[valid]\n        if len(values):\n            # Select an evenly distributed subset of pixels using linear spacing\n            ix = np.linspace(0, len(values) - 1, min(per_slice, len(values))).astype(int)\n            samples.append(values[ix])\n            \n    # 3. Validate that a sufficient number of valid pixels were collected across the series\n    if not samples or sum(map(len, samples)) < 32:\n        raise ValueError(\'Insufficient valid pixels\')\n        \n    # 4. Compute lower and upper percentiles (configured here as 0.5th and 99.5th) to establish robust intensity bounds\n    low, high = np.percentile(np.concatenate(samples), cfg[\'percentiles\'])\n    \n    # 5. Verify that the calculated limits are finite numbers and form a non-degenerate range\n    if not np.isfinite([low, high]).all() or high <= low:\n        raise ValueError(\'Constant/invalid series intensity\')\n        \n    return float(low), float(high)\n\n\ndef normalize_native_slice(image, valid, photo, low, high, matrix, offset, cfg):\n    """\n    Normalizes pixel intensities, handles photometric inversion, applies \n    conditional anti-aliasing for downsampling, and resamples the 2D slice \n    onto the standardized output grid using an affine transformation.\n    \n    Parameters:\n        image (np.ndarray): 32-bit floating-point raw slice image.\n        valid (np.ndarray): Boolean validity mask for the image pixels.\n        photo (str): Photometric interpretation (e.g., \'MONOCHROME1\' or \'MONOCHROME2\').\n        low (float): Lower intensity clipping threshold.\n        high (float): Upper intensity clipping threshold.\n        matrix (np.ndarray): 2D affine transformation matrix.\n        offset (np.ndarray): Affine translation offset.\n        cfg (dict): Pipeline configuration containing target size, antialiasing flags, etc.\n        \n    Returns:\n        tuple: (output_slice, support_mask)\n            - output_slice (np.ndarray): 16-bit floating-point normalized and resampled image.\n            - support_mask (np.ndarray): Boolean support mask indicating valid resampled pixels.\n    """\n    # 1. Scale pixel values into a [0, 1] range using global intensity limits and clip outliers\n    image = np.clip((image - low) / (high - low), 0, 1).astype(np.float32)\n    \n    # 2. Invert pixel intensities if photometric interpretation is MONOCHROME1 (lower values are intended to display brighter)\n    if photo == \'MONOCHROME1\':\n        image = 1 - image\n        \n    # 3. Force invalid or background padding pixels to zero\n    image[~valid] = 0\n    \n    # 4. Apply conditional anti-aliasing via Gaussian smoothing for in-plane reductions (downsampling)\n    #    Anti-alias only in-plane reductions; no filtering is ever applied along depth.\n    sigma = .5 * np.sqrt(np.maximum(np.sum(matrix ** 2, axis=1) - 1, 0))\n    if cfg[\'antialias\'] and sigma.max() > 0:\n        numerator = gaussian_filter(image, sigma, mode=\'constant\', cval=0)\n        denominator = gaussian_filter(valid.astype(np.float32), sigma, mode=\'constant\', cval=0)\n        image = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 1e-6)\n        \n    # 5. Resample the image and validity mask onto the standardized target grid using affine transformations\n    shape = (cfg[\'size\'], cfg[\'size\'])\n    output = affine_transform(image, matrix, offset, output_shape=shape, order=1, mode=\'constant\', cval=0, prefilter=False)\n    support = affine_transform(valid.astype(np.float32), matrix, offset, output_shape=shape, order=1, mode=\'constant\', cval=0, prefilter=False) > .999\n    \n    # 6. Zero out unsupported regions, ensure all values are finite, clip, and cast to float16 for memory efficiency\n    output[~support] = 0\n    if not np.isfinite(output).all():\n        raise ValueError(\'Nonfinite output\')\n        \n    return np.clip(output, 0, 1).astype(np.float16), support\n\n\ndef select_native_indices(n, depth):\n    """\n    Selects a uniform subset of slice indices from a 3D scan volume to match \n    a target depth, ensuring both ends of the volume are always included.\n    \n    Parameters:\n        n (int): Total number of available native slices in the series.\n        depth (int): Target number of slices required by the pipeline configuration.\n        \n    Returns:\n        np.ndarray: 1D array of unique integer slice indices.\n    """\n    # 1. If the total available slices are fewer than or equal to the target depth, return all indices\n    if n <= depth:\n        return np.arange(n, dtype=int)\n        \n    # 2. Otherwise, use linear spacing to select evenly distributed actual slices across the volume\n    #    (guaranteeing inclusion of both endpoints without artificial duplication or interpolation)\n    indices = np.rint(np.linspace(0, n - 1, depth)).astype(int)\n    \n    # 3. Assert that all selected indices are unique to prevent overlapping or repeated slices\n    assert len(np.unique(indices)) == depth\n    \n    return indices\n\n\ndef process_series(folder, study, series, plane, cfg):\n    """\n    Orchestrates the end-to-end preprocessing pipeline for a 3D DICOM series folder, \n    handling file discovery, fast header parsing, spatial validation, orientation \n    normalization, and cache array generation.\n    \n    Parameters:\n        folder (str or Path): Directory path containing the DICOM series files.\n        study (str): Study identifier.\n        series (str): Series identifier.\n        plane (str): Target anatomical viewing plane.\n        cfg (dict): Pipeline configuration dictionary.\n        \n    Returns:\n        tuple: (normalized image array, valid-pixel mask, metadata dictionary).\n    """\n    # 1. Discover and sort all DICOM files in the directory; raise an error if none are found\n    paths = sorted(Path(folder).glob(\'*.dcm\'))\n    if not paths:\n        raise FileNotFoundError(f\'No DICOM slices in {folder}\')\n        \n    # 2. Efficiently parse DICOM headers across all files without loading heavy pixel data into RAM\n    headers = [pydicom.dcmread(p, stop_before_pixels=True) for p in paths]\n    \n    # 3. Extract and validate spatial geometry and orientation from the headers\n    g = validate_and_extract_geometry(headers, study, series, plane, cfg)\n    \n    # 4. Reverse slice order if needed so the native normal has nonnegative projection on the listed plane normal\n    if g[\'basis\'][:, 0] @ AXES[plane][:, 0] < 0:\n        g[\'origin\'] = g[\'origin\'] + g[\'basis\'][:, 0] * (len(paths) - 1)\n        g[\'basis\'][:, 0] *= -1\n        g[\'order\'] = g[\'order\'][::-1]\n        \n    # 5. Compute transformation matrices, offsets, and cropping parameters for the native grid\n    matrix, offset, basis, crop = native_grid(g, plane, cfg)\n    \n    # 6. Reorder file paths based on the validated spatial sequence and select target depth slice indices\n    ordered = [paths[i] for i in g[\'order\']]\n    indices = select_native_indices(len(paths), cfg[\'depth\'])\n    \n    # 7. Execute the final slice-by-slice processing and build the native cache arrays\n    return build_native_cache_arrays(ordered, headers, g, indices, matrix, offset, basis, crop, cfg)\n\n\ndef build_native_cache_arrays(paths, headers, g, indices, matrix, offset, basis, crop, cfg):\n    """\n    Executes the streaming core processing loop with an estimated-memory guard for selected slices, normalizes \n    and resamples each slice onto the target grid, validates valid-pixel coverage, \n    and compiles a comprehensive metadata audit log.\n    \n    Parameters:\n        paths (list): Sorted list of file paths to DICOM slices.\n        headers (list): List of parsed DICOM dataset headers.\n        g (dict): Validated spatial geometry dictionary.\n        indices (np.ndarray): Selected target slice indices.\n        matrix (np.ndarray): 2D affine transformation matrix.\n        offset (np.ndarray): Affine translation offset.\n        basis (np.ndarray): Spatial basis vectors.\n        crop (np.ndarray): Fraction of source extent cropped along each output in-plane axis.\n        cfg (dict): Pipeline configuration dictionary.\n        \n    Returns:\n        tuple: (output_tensor, support_mask, audit_metadata)\n    """\n    # 1. Enforce a defensive memory guardrail by estimating working memory consumption\n    estimate = int(np.prod(g[\'shape\'][1:])) * 64 + cfg[\'depth\'] * cfg[\'size\'] ** 2 * 6 + cfg[\'intensity_sample_pixels\'] * 16\n    if estimate > cfg[\'max_working_memory_gb\'] * 1e9:\n        raise MemoryError(f\'Estimated streaming working memory {estimate / 1e9:.2f} GB exceeds configured guard\')\n        \n    # 2. Compute global intensity clipping bounds and pre-allocate target 3D tensors\n    low, high = series_intensity_limits(paths, cfg)\n    shape = (cfg[\'depth\'], cfg[\'size\'], cfg[\'size\'])\n    output = np.zeros(shape, np.float16)\n    support = np.zeros(shape, bool)\n    \n    # 3. Initialize tracking collections and normal vectors for spatial provenance\n    origins = []\n    positions = []\n    sops = []\n    normal = g[\'basis\'][:, 0] / np.linalg.norm(g[\'basis\'][:, 0])\n    \n    # 4. Iteratively decode, normalize, resample, and validate each selected native slice\n    for slot, index in enumerate(indices):\n        image, valid = decode_native_slice(paths[index])\n        output[slot], support[slot] = normalize_native_slice(image, valid, g[\'photo\'], low, high, matrix, offset, cfg)\n        \n        # Verify that the slice meets the minimum required valid-pixel coverage fraction (not anatomical segmentation)\n        if support[slot].mean() < cfg[\'min_valid_fraction\']:\n            raise ValueError(\'Too little support in a selected native slice\')\n            \n        # Compute physical LPS origin and relative position along the normal axis\n        origin = g[\'origin\'] + g[\'basis\'][:, 0] * int(index) + g[\'basis\'][:, 1:] @ offset\n        origins.append(origin.tolist())\n        positions.append(float((origin - g[\'origin\']) @ normal))\n        sops.append(str(headers[g[\'order\'][index]].SOPInstanceUID))\n        \n    # 5. Compute slice spacing gaps and assemble the comprehensive audit metadata dictionary\n    gaps = np.diff(positions)\n    meta = dict(\n        shape=list(shape),\n        input_shape=list(g[\'shape\']),\n        representation=\'native-plane-full-fov-v3\',\n        intensity_low=low,\n        intensity_high=high,\n        intensity_limits_method=\'bounded deterministic all-slice sample\',\n        source_sop_order=[str(headers[i].SOPInstanceUID) for i in g[\'order\']],\n        selected_sop_uids=sops,\n        selected_source_indices=indices.tolist(),\n        selected_count=len(indices),\n        padded_count=cfg[\'depth\'] - len(indices),\n        retained_slice_fraction=len(indices) / len(paths),\n        slice_positions_mm=positions,\n        slice_gap_min_mm=float(gaps.min()),\n        slice_gap_max_mm=float(gaps.max()),\n        spacing_summary_drc_mm=[float(np.median(gaps)), *np.linalg.norm(basis, axis=0).tolist()],\n        requested_inplane_spacing_mm=float(cfg[\'spacing_mm\']),\n        fov_spacing_adjusted=bool(np.linalg.norm(basis, axis=0).max() > cfg[\'spacing_mm\'] + 1e-8),\n        output_slice_origins_lps=origins,\n        output_inplane_basis_lps=basis.tolist(),\n        input_origin_lps=g[\'origin\'].tolist(),\n        input_basis_lps=g[\'basis\'].tolist(),\n        source_slice_spacing_mm=g[\'dz\'],\n        source_slice_thickness_mm=g[\'slice_thickness\'],\n        obliquity_degrees=g[\'angle\'],\n        review_warning=g[\'review_warning\'],\n        nearest_plane=g[\'nearest_plane\'],\n        crop_fraction_rc=crop.tolist(),\n        estimated_working_memory_gb=estimate / 1e9,\n        valid_fraction=float(support[:len(indices)].mean()),\n        tensor_valid_fraction=float(support.mean()),\n        transfer_syntaxes=sorted({str(h.file_meta.TransferSyntaxUID) for h in headers})\n    )\n    \n    return output, support, meta\n', 'transfer_label_split.py': '"""Verified holdout construction and teacher-label joining; no image/model imports."""\nfrom pathlib import Path\nimport hashlib\nimport numpy as np\nimport pandas as pd\n\nID = \'StudyInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n\n\ndef table(frame, name, binary=False, complete=False):\n    required = [ID, *TARGETS]\n    if set(required) - set(frame):\n        raise ValueError(f\'{name}: missing columns {set(required)-set(frame)}\')\n    out = frame[required].copy()\n    if out[ID].isna().any() or out[ID].astype(str).str.strip().eq(\'\').any() or out[ID].duplicated().any():\n        raise ValueError(f\'{name}: missing/duplicate study IDs\')\n    out[ID] = out[ID].astype(str)\n    for target in TARGETS:\n        out[target] = pd.to_numeric(out[target], errors=\'raise\')\n        known = out[target].notna()\n        values = out.loc[known, target].to_numpy(float)\n        if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():\n            raise ValueError(f\'{name}: invalid probabilities for {target}\')\n        if binary and not np.isin(values, [0, 1]).all():\n            raise ValueError(f\'{name}: verified targets must be binary\')\n        if complete and not known.all():\n            raise ValueError(f\'{name}: missing labels for {target}\')\n    return out.sort_values(ID).reset_index(drop=True)\n\n\ndef split_and_label(base, official, pseudo, radiologist, *, seed=42, gold_val_fraction=.3,\n                    pseudo_weight=.25, expected_gold=58, expected_radiologist=20):\n    """Hold out complete patient groups BEFORE joining any supervision.\n\n    Search deterministic candidate group splits using verified-label class counts,\n    never images, model predictions or validation scores. Validation is verified only.\n    """\n    if not 0 < gold_val_fraction < 1 or not 0 < pseudo_weight <= 1:\n        raise ValueError(\'Invalid fraction or pseudo weight\')\n    base = base.sort_values(ID).reset_index(drop=True).copy()\n    if base[ID].duplicated().any() or base.patient_group.isna().any():\n        raise ValueError(\'Invalid base study/group identities\')\n    if base.patient_group.astype(str).str.strip().eq(\'\').any():\n        raise ValueError(\'Empty patient group\')\n    official = table(official, \'official\', binary=True)\n    pseudo = table(pseudo, \'transferred labels\', complete=True)\n    radio = table(radiologist, \'radiologist\', binary=True, complete=True)\n    ids = set(base[ID])\n    if set(pseudo[ID]) != ids:\n        raise ValueError(\'Transferred label IDs must exactly match studies with images\')\n    if not ids <= set(official[ID]) or not set(radio[ID]) <= ids:\n        raise ValueError(\'Official/radiologist IDs do not match the image bundle\')\n    official = official.set_index(ID).loc[base[ID]].reset_index()\n    gold = official.loc[official[TARGETS].notna().any(axis=1)].copy()\n    if len(gold) != expected_gold or len(radio) != expected_radiologist:\n        raise ValueError(f\'Expected {expected_gold} gold and {expected_radiologist} radiologist studies; \'\n                         f\'found {len(gold)}, {len(radio)}\')\n    if set(gold[ID]) & set(radio[ID]):\n        raise ValueError(\'Official and radiologist cohorts overlap; resolve before splitting\')\n    groups = base.set_index(ID).patient_group\n    radio_groups = set(groups.loc[radio[ID]])\n    gold_groups = groups.loc[gold[ID]].to_numpy()\n    # Any official studies sharing a patient with the radiologist cohort must be held out.\n    mandatory = np.isin(gold_groups, list(radio_groups))\n    candidates = np.array(sorted(set(gold_groups) - radio_groups))\n    target_n = int(np.ceil(len(gold) * gold_val_fraction))\n    if mandatory.sum() >= len(gold) or not len(candidates):\n        raise ValueError(\'Cannot retain official gold training studies with these groups\')\n    pos = gold[TARGETS].eq(1).to_numpy(float)\n    neg = gold[TARGETS].eq(0).to_numpy(float)\n    counts = np.concatenate([pos, neg], axis=1)\n    total = counts.sum(axis=0)\n    rng = np.random.default_rng(seed)\n    best = None\n    for _ in range(2000):\n        held_groups = set(radio_groups)\n        for group in rng.permutation(candidates):\n            held = np.isin(gold_groups, list(held_groups))\n            if held.sum() >= target_n:\n                break\n            held_groups.add(group)\n        held = np.isin(gold_groups, list(held_groups))\n        if held.all() or not held.any():\n            continue\n        n = int(held.sum())\n        observed = counts[held].sum(axis=0)\n        # Size first, then avoid missing possible classes, then balance class counts.\n        missing = int(((total >= 2) & ((observed == 0) | (total-observed == 0))).sum())\n        score = (abs(n-target_n), missing,\n                 float(np.mean((observed-total*gold_val_fraction)**2/np.maximum(total, 1))))\n        if best is None or score < best[0]:\n            best = (score, held_groups, held.copy())\n    if best is None:\n        raise ValueError(\'Could not construct a valid gold group split\')\n    _, held_groups, gold_held = best\n    hold_ids = set(base.loc[base.patient_group.isin(held_groups), ID])\n    official_val_ids = set(gold.loc[gold_held, ID])\n    radio_ids = set(radio[ID])\n    # Preserve source metadata but replace ALL previous generated targets.\n    labels = base.copy().set_index(ID)\n    teacher = pseudo.set_index(ID).loc[labels.index]\n    truth = official.set_index(ID).loc[labels.index]\n    for target in TARGETS:\n        labels[target] = teacher[target]\n        labels[target+\'__mask\'] = 1\n        labels[target+\'__weight\'] = pseudo_weight\n        labels[target+\'__gold\'] = 0\n        verified = truth[target].notna()\n        labels.loc[verified, target] = truth.loc[verified, target]\n        labels.loc[verified, target+\'__weight\'] = 1.\n        labels.loc[verified, target+\'__gold\'] = 1\n    labels[\'validation_cohort\'] = \'\'\n    for cohort, frame in [(\'official\', gold.loc[gold_held]), (\'radiologist\', radio)]:\n        for _, row in frame.iterrows():\n            uid = row[ID]\n            labels.loc[uid, \'validation_cohort\'] = cohort\n            for target in TARGETS:\n                known = pd.notna(row[target])\n                # Clear teacher values at unknown validation targets.\n                labels.loc[uid, target] = row[target] if known else np.nan\n                labels.loc[uid, target+\'__mask\'] = int(known)\n                labels.loc[uid, target+\'__gold\'] = int(known)\n                labels.loc[uid, target+\'__weight\'] = float(known)\n    labels[\'has_gold\'] = labels[[t+\'__gold\' for t in TARGETS]].any(axis=1).astype(int)\n    labels[\'fold\'] = np.where(labels.index.isin(hold_ids), 1, 0)\n    labels = labels.reset_index()\n    train = labels.loc[~labels[ID].isin(hold_ids)].copy()\n    val = labels.loc[labels[ID].isin(official_val_ids | radio_ids)].copy()\n    assert not set(train[ID]) & set(val[ID])\n    assert not set(train.patient_group) & set(val.patient_group)\n    assert not set(train[ID]) & radio_ids\n    split = labels[[ID, \'patient_group\', \'validation_cohort\']].copy()\n    split[\'role\'] = np.where(split[ID].isin(train[ID]), \'training\',\n                            np.where(split[ID].isin(val[ID]), \'validation\', \'held_group_unused\'))\n    summary = dict(training_studies=len(train), validation_studies=len(val),\n                   official_gold_training=int(train[ID].isin(set(gold[ID])).sum()),\n                   official_gold_validation=len(official_val_ids), radiologist_validation=len(radio),\n                   excluded_group_companions=int(split.role.eq(\'held_group_unused\').sum()),\n                   split_policy=\'group-disjoint-label-balanced-70-30-v1\', seed=seed)\n    return train, val, split, summary\n\n\ndef label_counts(frame):\n    rows = []\n    for target in TARGETS:\n        known = frame[target+\'__gold\'].eq(1)\n        rows.append(dict(target=target, known=int(known.sum()),\n                         positive=int((known & frame[target].eq(1)).sum()),\n                         negative=int((known & frame[target].eq(0)).sum())))\n    return pd.DataFrame(rows)\n', 'plane_fs_attention.py': '"""Percentage sampling and condition-specific attention over six plane/FS groups."""\nimport math\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\nfrom torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights\n\nGROUPS = [(\'Sagittal\', 1), (\'Sagittal\', 0), (\'Coronal\', 1),\n          (\'Coronal\', 0), (\'Axial\', 1), (\'Axial\', 0)]\n\n\ndef group_index(record):\n    plane = str(record[\'Anatomical_Plane\']).strip().capitalize()\n    fs = record[\'Fat_Suppression\']\n    if fs not in (0, 1):\n        raise ValueError(f\'Unknown fat suppression: {fs!r}\')\n    try:\n        return GROUPS.index((plane, int(fs)))\n    except ValueError as exc:\n        raise ValueError(f\'Unknown anatomical plane: {plane!r}\') from exc\n\n\ndef sample_triplet_indices(n, cfg, rng=None):\n    """Select unique triplet starts; never sample depth padding.\n\n    Percentages refer to retained real slices, not discarded original DICOM slices.\n    Short stacks use every possible unique triplet if fewer than the requested minimum.\n    """\n    if not isinstance(n, (int, np.integer)) or n < 1:\n        raise ValueError(\'Positive integer real depth required\')\n    fraction = float(cfg[\'sample_fraction\'])\n    minimum, maximum = cfg[\'min_triplets\'], cfg[\'max_triplets\']\n    emphasis = float(cfg[\'central_sample_fraction\'])\n    width = float(cfg[\'central_depth_fraction\'])\n    if (not 0 < fraction <= 1 or not 0 < width < 1 or not 0 < emphasis < 1\n            or not isinstance(minimum, int) or not isinstance(maximum, int)\n            or not 1 <= minimum <= maximum):\n        raise ValueError(\'Invalid percentage-sampling configuration\')\n    if n < 3:\n        return np.clip(np.array([[0, 1, 2]]), 0, n-1)\n    count = min(max(minimum, math.ceil(n*fraction)), maximum, n-2)\n    starts = np.arange(n-2)\n    centers = (starts + 1.5) / n\n    lo, hi = .5-width/2, .5+width/2\n    central = starts[(centers >= lo) & (centers < hi)]\n    outer = starts[(centers < lo) | (centers >= hi)]\n    # Retain at least two outer samples where possible to cover both ends.\n    central_count = int(round(count*emphasis))\n    if len(outer) >= 2 and count >= 4:\n        central_count = min(central_count, count-2)\n    central_count = max(count-len(outer), min(central_count, len(central)))\n    outer_count = count-central_count\n\n    def choose(pool, k):\n        if not k:\n            return np.array([], dtype=int)\n        if rng is not None:\n            return rng.choice(pool, size=k, replace=False)\n        # Interior quantiles are deterministic and unique when k <= len(pool).\n        indices = np.floor((np.arange(k)+.5)*len(pool)/k).astype(int)\n        return pool[indices]\n\n    left = outer[(outer+1.5)/n < .5]\n    right = outer[(outer+1.5)/n >= .5]\n    left_count = outer_count//2\n    if outer_count % 2 and rng is not None:\n        left_count += int(rng.integers(0, 2))\n    left_count = min(len(left), left_count)\n    right_count = min(len(right), outer_count-left_count)\n    left_count = min(len(left), outer_count-right_count)\n    picked = np.sort(np.concatenate([choose(central, central_count),\n        choose(left, left_count), choose(right, right_count)]))\n    assert len(np.unique(picked)) == count\n    return picked[:, None] + np.arange(3)\n\n\nclass GroupAttentionHead(nn.Module):\n    def __init__(self, dropout=.2, dim=1536, hidden=128, targets=12):\n        super().__init__()\n        self.projection = nn.Linear(dim, hidden)\n        self.group_embedding = nn.Embedding(6, hidden)\n        self.score = nn.Linear(hidden, targets)\n        self.classifier = nn.Linear(dim, targets)\n        self.dropout = nn.Dropout(dropout)\n        # Start with equal attention over PRESENT groups; learn condition-specific weights.\n        nn.init.zeros_(self.score.weight)\n        nn.init.zeros_(self.score.bias)\n\n    def forward(self, vectors, group_ids, return_attention=False):\n        if vectors.ndim != 2 or vectors.shape[0] == 0 or vectors.shape[1] != self.classifier.in_features:\n            raise ValueError(\'Expected nonempty series feature matrix\')\n        ids = torch.as_tensor(group_ids, dtype=torch.long, device=vectors.device)\n        if ids.shape != (len(vectors),) or ((ids < 0) | (ids >= 6)).any():\n            raise ValueError(\'Invalid group IDs\')\n        sums = vectors.new_zeros(6, vectors.shape[1]).index_add(0, ids, vectors)\n        counts = torch.bincount(ids, minlength=6)\n        present = counts > 0\n        groups = sums / counts.clamp_min(1).to(vectors.dtype).unsqueeze(1)\n        hidden = torch.tanh(self.projection(groups) + self.group_embedding.weight)\n        scores = self.score(hidden).float().transpose(0, 1)\n        scores = scores.masked_fill(~present.unsqueeze(0), float(\'-inf\'))\n        attention = scores.softmax(dim=1)\n        fused = attention.to(groups.dtype) @ groups\n        logits = (self.dropout(fused)*self.classifier.weight).sum(dim=1) + self.classifier.bias\n        logits = logits.unsqueeze(0)\n        return (logits, attention, present) if return_attention else logits\n\n\nclass ConvNeXtKnee(nn.Module):\n    def __init__(self, pretrained=False, weights_path=None, dropout=.2, attention_dim=128):\n        super().__init__()\n        self.backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained and weights_path is None else None)\n        if weights_path:\n            self.backbone.load_state_dict(torch.load(weights_path, map_location=\'cpu\', weights_only=True), strict=True)\n        self.backbone.classifier[2] = nn.Identity()  # Preserve pretrained LayerNorm and flatten.\n        self.head = GroupAttentionHead(dropout=dropout, hidden=attention_dim)\n        self.register_buffer(\'mean\', torch.tensor([.485,.456,.406]).view(1,3,1,1))\n        self.register_buffer(\'std\', torch.tensor([.229,.224,.225]).view(1,3,1,1))\n\n    def forward(self, series, microbatch=8, return_attention=False):\n        if not series:\n            raise ValueError(\'A study must have at least one valid series\')\n        if microbatch < 1:\n            raise ValueError(\'microbatch must be positive\')\n        vectors, ids = [], []\n        for item in series:\n            images = item[\'images\']\n            if len(images) == 0:\n                raise ValueError(\'Empty series\')\n            features = []\n            for start in range(0, len(images), microbatch):\n                x = images[start:start+microbatch].to(self.mean.device)\n                x = (x-self.mean)/self.std\n                f = checkpoint(self.backbone, x, use_reentrant=False) if self.training else self.backbone(x)\n                features.append(f.float())\n            f = torch.cat(features)\n            vectors.append(torch.cat([f.mean(0), f.max(0).values]))\n            ids.append(item[\'group\'])\n        return self.head(torch.stack(vectors), ids, return_attention=return_attention)\n\n\n'}
CODE_ROOT = Path('/kaggle/working/_convnext_tiny_central_group_attention_code')
CODE_ROOT.mkdir(parents=True, exist_ok=True)
for name, source in SOURCE_FILES.items(): (CODE_ROOT/name).write_text(source)
sys.path.insert(0,str(CODE_ROOT))
for name in ('knee_data','cnn_runtime','plane_fs_attention','transfer_label_split'):
    sys.modules.pop(name,None)
import knee_data as data
import cnn_runtime as cnn
import plane_fs_attention as attention
import transfer_label_split as transfer
import numpy as np
import pandas as pd


## Verify labels, holdout and initialization


In [3]:
from copy import deepcopy
import os

if min(CFG[k] for k in ['microbatch', 'accumulate', 'min_triplets', 'max_triplets', 'epochs', 'patience']) < 1:
    raise ValueError('Training sizes and patience must be positive')
torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed']); random.seed(CFG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = data.discover_bundle(DATASET_ROOT, dataset_handle=DATASET_HANDLE)
if data.file_hash(ROOT/'dataset_index.json') != EXPECTED_SOURCE_INDEX_SHA256:
    raise ValueError('Image index differs from the source used to generate transferred labels')
print('Loading image manifests:', ROOT, flush=True)
SERIES, BASE_LABELS, ALL_LABELS, INDEX = data.load_bundle(ROOT)
if INDEX['preprocessing'].get('implementation') != 'ef1a2effbba938df32d512f010e3786904efc0cda3d04c95b074e29265018b5d':
    raise ValueError('Unexpected preprocessing implementation')
planes = SERIES['Anatomical_Plane'].astype(str).str.strip().str.capitalize()
if not planes.isin(['Sagittal', 'Coronal', 'Axial']).all():
    raise ValueError('Unknown plane metadata')
if COMPETITION_ROOT is None:
    COMPETITION_ROOT = data.select_one(
        [p.parent for p in data.find_input_files('/kaggle/input', 'train.csv')
         if (p.parent / 'train_series.csv').is_file()], 'COMPETITION_ROOT')
OFFICIAL_PATH = Path(COMPETITION_ROOT) / 'train.csv'
official = transfer.table(pd.read_csv(OFFICIAL_PATH, dtype={data.ID: str}), 'official', binary=True)
if set(official[data.ID]) != set(ALL_LABELS[data.ID]):
    raise ValueError('Official study IDs differ from source bundle')
original = official.set_index(data.ID).loc[ALL_LABELS[data.ID]]
for target in data.TARGETS:
    known = original[target].notna().to_numpy()
    if not np.array_equal(known, ALL_LABELS[target+'__gold'].to_numpy(bool)):
        raise ValueError('Official mask drift: '+target)
    if not np.array_equal(original[target].to_numpy()[known], ALL_LABELS[target].to_numpy()[known]):
        raise ValueError('Official label drift: '+target)
if TRANSFER_LABELS_PATH is None:
    TRANSFER_LABELS_PATH = ROOT / 'transferred_labels.csv'
TRANSFER_LABELS_PATH = Path(TRANSFER_LABELS_PATH)
if not TRANSFER_LABELS_PATH.is_file():
    raise FileNotFoundError('Use the image dataset version containing transferred_labels.csv')
if RADIOLOGIST_LABELS_PATH is None:
    candidates = []
    for current, dirs, files in os.walk('/kaggle/input'):
        dirs[:] = [d for d in dirs if d not in {'train_series', 'test_series', 'cache'}]
        if 'rsna-knee-radiologist-verified-20' not in Path(current).parts:
            continue
        for name in files:
            if name.endswith('.csv'):
                path = Path(current)/name
                header = pd.read_csv(path, nrows=0)
                if {data.ID, *data.TARGETS} <= set(header):
                    candidates.append(path)
    RADIOLOGIST_LABELS_PATH = data.select_one(candidates, 'RADIOLOGIST_LABELS_PATH')
RADIOLOGIST_LABELS_PATH = Path(RADIOLOGIST_LABELS_PATH)
pseudo = pd.read_csv(TRANSFER_LABELS_PATH, dtype={data.ID: str})
radio = pd.read_csv(RADIOLOGIST_LABELS_PATH, dtype={data.ID: str})
TRAIN, VAL, SPLIT, SPLIT_SUMMARY = transfer.split_and_label(
    BASE_LABELS, official, pseudo, radio, seed=CFG['seed'],
    gold_val_fraction=CFG['gold_val_fraction'], pseudo_weight=CFG['pseudo_weight'],
    expected_gold=EXPECTED_OFFICIAL_GOLD_STUDIES,
    expected_radiologist=EXPECTED_RADIOLOGIST_STUDIES)
data.validate_labels(TRAIN); data.validate_labels(VAL)
if set(TRAIN[data.ID]) & set(VAL[data.ID]) or set(TRAIN.patient_group) & set(VAL.patient_group):
    raise ValueError('Training/validation leakage')
LABEL_PROVENANCE = dict(transferred_labels_sha256=data.file_hash(TRANSFER_LABELS_PATH),
    source_index_sha256=EXPECTED_SOURCE_INDEX_SHA256,
    official_sha256=data.file_hash(OFFICIAL_PATH),
    radiologist_sha256=data.file_hash(RADIOLOGIST_LABELS_PATH),
    teacher_probabilities='raw probabilities; no extra smoothing; not OOF',
    teacher_sha256='7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef',
    transfer_runtime_sha256='48d7dcecd9d651827f33adacbe435890f746926990197a58cf32e5d7dc9500c2',
    teacher_model_sha256='363f56254921758a7cd76c650d1dd19a44f64a2a4d4623dbc161a84b4a826e4e',
    recipe='normalized-to-raptor-v1;raw-sigmoid;plain-mirror-mean',
    verified_weight=1., pseudo_weight=CFG['pseudo_weight'],
    loss_normalization=CFG['loss_normalization'])
# Validate metadata before any training; unknown FS is not silently mapped to non-FS.
for plane, fs in SERIES[['Anatomical_Plane','Fat_Suppression']].drop_duplicates().itertuples(index=False,name=None):
    attention.group_index(dict(Anatomical_Plane=plane,Fat_Suppression=fs))
for n in (1,2,3,4,10,20,30,64):
    triplets=cnn.sample_triplet_indices(n,CFG)
    print('Retained depth',n,'triplets',len(triplets),triplets+1,flush=True)
split_hash=data.fingerprint(SPLIT.to_dict('records'))
INITIALIZATION=dict(mode='imagenet-convnext-tiny-plane-fs-attention-v1',
    backbone='torchvision ConvNeXt-Tiny IMAGENET1K_V1' if PRETRAINED_PATH is None else data.file_hash(PRETRAINED_PATH))
IDENTITY=dict(configuration=deepcopy(CFG),dataset=INDEX['identity'],label_provenance=LABEL_PROVENANCE,
    split_hash=split_hash,source_hash=data.fingerprint(SOURCE_FILES),initialization=INITIALIZATION)
WORK=OUTPUT_ROOT/data.fingerprint(IDENTITY)[:16]
WORK.mkdir(parents=True,exist_ok=True)
SPLIT.to_csv(WORK/'split_private.csv',index=False)
data.atomic_json(WORK/'run_identity.json',IDENTITY)
data.atomic_json(WORK/'initialization.json',INITIALIZATION)
data.atomic_json(WORK/'label_provenance.json',LABEL_PROVENANCE)
data.atomic_json(WORK/'split_summary.json',SPLIT_SUMMARY)
for cohort,frame in [('training',TRAIN),('combined',VAL),
                     ('official',VAL.loc[VAL.validation_cohort.eq('official')]),
                     ('radiologist',VAL.loc[VAL.validation_cohort.eq('radiologist')])]:
    counts=transfer.label_counts(frame)
    counts.to_csv(WORK/(cohort+'_verified_counts.csv'),index=False)
    print(cohort,'studies:',len(frame));display(counts)
print('Split:',SPLIT_SUMMARY,'Initialization:',INITIALIZATION,'Output:',WORK,flush=True)
MODEL=cnn.ConvNeXtKnee(pretrained=RESUME_CHECKPOINT is None,
    weights_path=PRETRAINED_PATH if RESUME_CHECKPOINT is None else None,
    dropout=CFG['dropout'],attention_dim=CFG['attention_dim']).to(DEVICE)
INITIAL_BEST=None
if RESUME_CHECKPOINT is not None:
    saved=torch.load(RESUME_CHECKPOINT,map_location='cpu',weights_only=False)
    if saved['identity'] != IDENTITY:raise ValueError('Exact resume identity mismatch')
    print('Resume epoch:',saved['epoch']+1,'Bad epochs:',saved['bad_epochs'],flush=True)
    del saved
else:
    print('Fresh ImageNet backbone; new attention/classifier; training starts at epoch 1.',flush=True)



Loading image manifests: /kaggle/input/datasets/gany24558/rsna-knee-normalized-all-training-data


Retained depth 1 triplets 1 [[1 1 1]]


Retained depth 2 triplets 1 [[1 2 2]]


Retained depth 3 triplets 1 [[1 2 3]]


Retained depth 4 triplets 2 [[1 2 3]
 [2 3 4]]


...

 59%|█████▉    | 64.8M/109M [00:00<00:00, 202MB/s]

 77%|███████▋  | 84.1M/109M [00:00<00:00, 200MB/s]

 95%|█████████▍| 103M/109M [00:00<00:00, 197MB/s] 

100%|██████████| 109M/109M [00:00<00:00, 202MB/s]

Fresh ImageNet backbone; new attention/classifier; training starts at epoch 1.


## Train


In [4]:
BEST,HISTORY,STOP_REASON=cnn.train_model(MODEL,TRAIN,VAL,SERIES,INDEX['identity']['run_id'],
    CFG,WORK,IDENTITY,RESUME_CHECKPOINT,initial_best=INITIAL_BEST)
print('Stop reason:',STOP_REASON,'Best epoch:',BEST['epoch'],'Validation BCE:',BEST['val_loss'])
display(pd.DataFrame(HISTORY))
data.atomic_json(WORK/'best_validation_metrics.json',BEST['metrics'])
for cohort,metrics in BEST['metrics'].get('cohorts',{}).items():
    print(cohort,'BCE:',metrics['bce'],'AUROC:',metrics['macro_auroc'])
    display(pd.DataFrame(metrics['targets']))
print('SAVE this full checkpoint before ending the session:',WORK/'last.pt',flush=True)


Epoch 1: 100/4369 studies


Epoch 1: 200/4369 studies


Epoch 1: 300/4369 studies


Epoch 1: 400/4369 studies


Epoch 1: 500/4369 studies


...

Epoch 2: 2700/4369 studies


Epoch 2: 2800/4369 studies


Epoch 2: 2900/4369 studies


Epoch 2: 3000/4369 studies


Epoch 2: 3100/4369 studies


## Inspect selected model attention


In [ ]:
# Attention is relative importance, not diagnostic confidence.
MODEL.eval()
by_study={uid:g.sort_values(data.SID).to_dict('records') for uid,g in SERIES.groupby(data.ID)}
rows=[]
with torch.inference_mode():
    for row in VAL.to_dict('records'):
        bag=cnn.study_images(row[data.ID],by_study,INDEX['identity']['run_id'],CFG)
        with torch.autocast(DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=='cuda'):
            _,weights,present=MODEL(bag,CFG['microbatch'],return_attention=True)
        for t,target in enumerate(data.TARGETS):
            for g,(plane,fs) in enumerate(attention.GROUPS):
                rows.append({data.ID:row[data.ID],'cohort':row['validation_cohort'],'target':target,
                    'plane':plane,'fat_suppression':fs,'present':bool(present[g]),
                    'attention':float(weights[t,g])})
attention_table=pd.DataFrame(rows)
attention_table.to_csv(WORK/'validation_attention_private.csv',index=False)
display(attention_table.groupby(['target','plane','fat_suppression']).attention.mean().unstack(['plane','fat_suppression']))


## Export and reload-test


In [ ]:
PACKAGE=cnn.export_model(MODEL,BEST,CFG,INDEX,IDENTITY,WORK,SOURCE_FILES)
print('Exported and reload-tested:',PACKAGE)


## Publish separate model variation


In [ ]:
if UPLOAD_MODEL:
    import kagglehub
    manifest=json.loads((PACKAGE/'manifest.json').read_text())
    for name,digest in manifest['files'].items():
        if data.file_hash(PACKAGE/name)!=digest:raise ValueError('Export changed: '+name)
    kagglehub.model_upload(MODEL_HANDLE,str(PACKAGE),version_notes=(
        f"Percentage central sampling; six plane/FS groups; condition attention; "
        f"transferred labels; best epoch {BEST['epoch']}; verified BCE {BEST['val_loss']:.6f}"))
    print('Upload accepted: https://www.kaggle.com/models/'+MODEL_HANDLE)
else:print('Upload disabled:',PACKAGE)
